# Qwen2 Model Distillation Notebook

This notebook distills knowledge from a teacher model into three Qwen2 student models:
1. **Qwen2-0.5B** (smallest, trained first)
2. **Qwen2-1.5B** (medium)
3. **Qwen2-7B** (largest, trained last)

> **Note:** There is no official Qwen2-4B model. If you need a 4B model, consider using Qwen2.5-3B instead.

This notebook reuses functions from `distil_logits.py` for consistency and maintainability.

## 1. Imports and Environment Setup

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
import yaml
import gc

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Reusable Functions from distil_logits.py

These functions are imported/adapted from the original `distil_logits.py` script.

In [ ]:
def sharegpt_format(example, tokenizer, chat_template):
    """
    Convert ShareGPT format conversations to chat format.
    Adapted from distil_logits.py
    """
    conversations = example['conversations']
    message = []
    
    if isinstance(conversations, list):
        for conversation in conversations:
            if isinstance(conversation, dict):
                if conversation.get('from') == 'human':
                    message.append({"role": "user", "content": conversation.get('value', '')})
                elif conversation.get('from') == 'gpt':
                    message.append({"role": "assistant", "content": conversation.get('value', '')})
                elif conversation.get('from') == 'system':
                    message.insert(0, {"role": "system", "content": conversation.get('value', '')})

    if not any(msg.get('role') == 'system' for msg in message):
        message.insert(0, {"role": "system", "content": "You are a helpful assistant."})

    # Apply chat template
    tokenizer.chat_template = chat_template
    text = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    return {"text": text}


def tokenize_function(examples, tokenizer, max_length):
    """
    Tokenize examples with truncation and padding.
    Adapted from distil_logits.py
    """
    return tokenizer(examples["text"], truncation=True, max_length=max_length, padding="max_length")


def pad_logits(student_logits, teacher_logits):
    """
    Pad logits to match vocabulary sizes between student and teacher models.
    Directly from distil_logits.py
    """
    student_size, teacher_size = student_logits.size(-1), teacher_logits.size(-1)
    if student_size != teacher_size:
        pad_size = abs(student_size - teacher_size)
        pad_tensor = torch.zeros((*teacher_logits.shape[:-1], pad_size), dtype=teacher_logits.dtype, device=teacher_logits.device)
        return (torch.cat([student_logits, pad_tensor], dim=-1), teacher_logits) if student_size < teacher_size else (student_logits, torch.cat([teacher_logits, pad_tensor], dim=-1))
    return student_logits, teacher_logits


def freeze_student_spectrum(model, unfrozen_layers_file):
    """
    Freeze layers of the student model based on spectrum configuration.
    Directly from distil_logits.py
    """
    with open(unfrozen_layers_file, 'r') as file:
        unfrozen_layers = yaml.safe_load(file)['unfrozen_parameters']
    
    for name, param in model.named_parameters():
        if not any(layer in name for layer in unfrozen_layers):
            param.requires_grad = False
        else:
            param.requires_grad = True

In [ ]:
class LogitsTrainer(SFTTrainer):
    """
    Custom trainer for logit-based knowledge distillation.
    Directly from distil_logits.py with parameterized distillation settings.
    """
    def __init__(self, *args, temperature=2.0, alpha=0.5, max_length=4096, **kwargs):
        super().__init__(*args, **kwargs)
        self.temperature = temperature
        self.alpha = alpha
        self.max_length = max_length
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        device = next(model.parameters()).device
        inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}
        self.teacher_model = self.teacher_model.to(device)
        
        student_model = model.module if hasattr(model, 'module') else model
        teacher_model = self.teacher_model.module if hasattr(self.teacher_model, 'module') else self.teacher_model

        student_outputs = student_model(**inputs)
        with torch.no_grad():
            teacher_outputs = teacher_model(**inputs)

        custom_loss = self.distillation_loss(model, student_outputs.logits, teacher_outputs.logits, inputs, student_outputs.loss)
        return (custom_loss, student_outputs) if return_outputs else custom_loss

    def distillation_loss(self, model, student_logits, teacher_logits, inputs, original_loss):
        device = next(model.parameters()).device
        student_logits, teacher_logits = pad_logits(student_logits.to(device), teacher_logits.to(device))
        
        student_logits_scaled = student_logits / self.temperature
        teacher_logits_scaled = teacher_logits / self.temperature

        loss_kd = F.kl_div(
            F.log_softmax(student_logits_scaled, dim=-1),
            F.softmax(teacher_logits_scaled, dim=-1),
            reduction='batchmean'
        ) * (self.temperature ** 2) / self.max_length

        return self.alpha * loss_kd + (1 - self.alpha) * original_loss

## 3. Helper Functions for Multi-Model Distillation

In [ ]:
def load_and_prepare_dataset(dataset_config, tokenizer, chat_template, max_length, num_proc=8):
    """
    Load and prepare dataset for training.
    """
    print(f"Loading dataset: {dataset_config['name']}")
    dataset = load_dataset(dataset_config["name"], split=dataset_config["split"])
    dataset = dataset.shuffle(seed=dataset_config["seed"])
    
    if "num_samples" in dataset_config and dataset_config["num_samples"]:
        dataset = dataset.select(range(dataset_config["num_samples"]))
        print(f"Using {dataset_config['num_samples']} samples")
    
    # Preprocess dataset
    print("Preprocessing and tokenizing dataset...")
    original_columns = dataset.column_names
    
    # Apply sharegpt format
    dataset = dataset.map(
        lambda x: sharegpt_format(x, tokenizer, chat_template),
        remove_columns=original_columns
    )
    
    # Tokenize
    tokenized_dataset = dataset.map(
        lambda x: tokenize_function(x, tokenizer, max_length),
        batched=True,
        num_proc=num_proc,
        remove_columns=["text"]
    )
    
    # Split into train/test
    tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)
    print(f"Dataset prepared: {len(tokenized_dataset['train'])} train, {len(tokenized_dataset['test'])} test samples")
    
    return tokenized_dataset


def load_model(model_name, use_flash_attention=True):
    """
    Load a model with optional flash attention.
    """
    print(f"Loading model: {model_name}")
    model_kwargs = {"torch_dtype": torch.bfloat16}
    if use_flash_attention:
        model_kwargs["attn_implementation"] = "flash_attention_2"
    
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    print(f"Model loaded: {model_name}")
    return model


def cleanup_memory():
    """
    Clean up GPU memory between training runs.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("Memory cleaned up")

In [ ]:
def train_student_model(student_model_name, teacher_model, tokenized_dataset, config):
    """
    Train a single student model using knowledge distillation.
    """
    print(f"\n{'='*60}")
    print(f"Training Student Model: {student_model_name}")
    print(f"{'='*60}\n")
    
    # Load student model
    student_model = load_model(
        student_model_name,
        use_flash_attention=config["model_config"]["use_flash_attention"]
    )
    
    # Apply spectrum freezing if configured
    if "spectrum" in config and "layers_to_unfreeze" in config.get("spectrum", {}):
        print("Applying Spectrum layer freezing...")
        freeze_student_spectrum(student_model, config["spectrum"]["layers_to_unfreeze"])
    else:
        print("All layers of the student model will be trainable.")
    
    # Set up output directory for this model
    model_short_name = student_model_name.split('/')[-1]
    output_dir = os.path.join(config["training"]["output_dir"], model_short_name)
    
    # Update training arguments for this model
    training_config = config["training"].copy()
    training_config["output_dir"] = output_dir
    training_arguments = TrainingArguments(**training_config)
    
    # Create trainer
    trainer = LogitsTrainer(
        model=student_model,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        args=training_arguments,
        temperature=config["distillation"]["temperature"],
        alpha=config["distillation"]["alpha"],
        max_length=config["tokenizer"]["max_length"]
    )
    
    # Add teacher model to trainer
    trainer.teacher_model = teacher_model
    
    # Train
    print(f"Starting training for {student_model_name}...")
    trainer.train(resume_from_checkpoint=config["training"]["resume_from_checkpoint"])
    
    # Save the final model
    trainer.save_model(output_dir)
    print(f"Model saved to: {output_dir}")
    
    # Cleanup
    del student_model
    del trainer
    cleanup_memory()
    
    return output_dir

## 4. Configuration

Define the configuration for distillation. Modify these settings as needed.

In [ ]:
# Main configuration
config = {
    "project_name": "distil-qwen2-multi",
    "dataset": {
        "name": "mlabonne/FineTome-100k",
        "split": "train",
        "num_samples": 10000,  # Limit samples for faster training; set to None for full dataset
        "seed": 42
    },
    "models": {
        "teacher": "arcee-ai/Arcee-Spark",
        # Student models ordered from smallest to largest
        "students": [
            "Qwen/Qwen2-0.5B",   # Smallest - trained first
            "Qwen/Qwen2-1.5B",   # Medium
            "Qwen/Qwen2-7B"      # Largest - trained last (no 4B exists)
        ]
    },
    "tokenizer": {
        "max_length": 4096,
        "chat_template": "{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n' }}{% endif %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
    },
    "training": {
        "output_dir": "./results",
        "num_train_epochs": 3,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "save_steps": 1000,
        "logging_steps": 10,
        "learning_rate": 2e-5,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "resume_from_checkpoint": None,
        "fp16": False,
        "bf16": True
    },
    "distillation": {
        "temperature": 2.0,
        "alpha": 0.5
    },
    "model_config": {
        "use_flash_attention": True
    }
    # Uncomment to use Spectrum layer freezing:
    # "spectrum": {
    #     "layers_to_unfreeze": "/path/to/spectrum_config.yaml"
    # }
}

# Set wandb project
os.environ['WANDB_PROJECT'] = config["project_name"]

print("Configuration loaded:")
print(f"  Teacher model: {config['models']['teacher']}")
print(f"  Student models: {config['models']['students']}")
print(f"  Dataset: {config['dataset']['name']}")
print(f"  Max length: {config['tokenizer']['max_length']}")

## 5. Load Teacher Model and Prepare Dataset

Load the teacher model once and reuse it for all student models.

In [ ]:
# Load teacher model (shared across all distillations)
print("Loading teacher model...")
teacher_model = load_model(
    config["models"]["teacher"],
    use_flash_attention=config["model_config"]["use_flash_attention"]
)
teacher_model.eval()  # Set to evaluation mode

# Freeze teacher model parameters
for param in teacher_model.parameters():
    param.requires_grad = False

print(f"Teacher model loaded and frozen: {config['models']['teacher']}")

In [ ]:
# Load tokenizer (use teacher tokenizer for consistency)
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config["models"]["teacher"])

# Prepare dataset
tokenized_dataset = load_and_prepare_dataset(
    config["dataset"],
    tokenizer,
    config["tokenizer"]["chat_template"],
    config["tokenizer"]["max_length"]
)

## 6. Sequential Distillation: Smallest to Largest

Train each student model sequentially, starting from the smallest (Qwen2-0.5B).

### 6.1 Distill into Qwen2-0.5B (Smallest)

In [ ]:
# Train Qwen2-0.5B (smallest model)
student_0_5b = config["models"]["students"][0]
print(f"\nStarting distillation for: {student_0_5b}")

output_0_5b = train_student_model(
    student_model_name=student_0_5b,
    teacher_model=teacher_model,
    tokenized_dataset=tokenized_dataset,
    config=config
)

print(f"\nQwen2-0.5B distillation complete! Model saved to: {output_0_5b}")

### 6.2 Distill into Qwen2-1.5B (Medium)

In [ ]:
# Train Qwen2-1.5B (medium model)
student_1_5b = config["models"]["students"][1]
print(f"\nStarting distillation for: {student_1_5b}")

output_1_5b = train_student_model(
    student_model_name=student_1_5b,
    teacher_model=teacher_model,
    tokenized_dataset=tokenized_dataset,
    config=config
)

print(f"\nQwen2-1.5B distillation complete! Model saved to: {output_1_5b}")

### 6.3 Distill into Qwen2-7B (Largest)

In [ ]:
# Train Qwen2-7B (largest model)
student_7b = config["models"]["students"][2]
print(f"\nStarting distillation for: {student_7b}")

output_7b = train_student_model(
    student_model_name=student_7b,
    teacher_model=teacher_model,
    tokenized_dataset=tokenized_dataset,
    config=config
)

print(f"\nQwen2-7B distillation complete! Model saved to: {output_7b}")

## 7. Summary and Cleanup

In [ ]:
# Final cleanup
del teacher_model
cleanup_memory()

# Summary
print("\n" + "="*60)
print("DISTILLATION COMPLETE")
print("="*60)
print(f"\nDistilled models saved to:")
for student in config["models"]["students"]:
    model_name = student.split('/')[-1]
    output_path = os.path.join(config["training"]["output_dir"], model_name)
    print(f"  - {student}: {output_path}")

print(f"\nTeacher model: {config['models']['teacher']}")
print(f"Dataset: {config['dataset']['name']}")
print(f"Training epochs: {config['training']['num_train_epochs']}")
print(f"Distillation temperature: {config['distillation']['temperature']}")
print(f"Distillation alpha: {config['distillation']['alpha']}")

## 8. (Optional) Run All Models Sequentially

Alternatively, use this cell to run all distillations in a single loop.

In [ ]:
# Uncomment and run this cell to train all models in a loop
# (Only use if you haven't run sections 6.1-6.3)

# results = {}
# for student_model_name in config["models"]["students"]:
#     output_path = train_student_model(
#         student_model_name=student_model_name,
#         teacher_model=teacher_model,
#         tokenized_dataset=tokenized_dataset,
#         config=config
#     )
#     results[student_model_name] = output_path
#     print(f"\n{'='*40}")
#     print(f"Completed: {student_model_name}")
#     print(f"Saved to: {output_path}")
#     print(f"{'='*40}\n")

# print("\nAll distillations complete!")
# for model, path in results.items():
#     print(f"  {model}: {path}")